# Week 2: Probing Classifier + Exp 1 (Layer-wise AUROC)

**목표**
1. Week 1에서 Drive에 저장한 hidden states 로드
2. 레이어별 Probing Classifier(LogisticRegression) 학습
3. AUROC 히트맵 생성 → 최적 레이어 선정
4. `config.py`에 `BEST_LAYER` 업데이트

**실행 환경**: Google Colab Pro (A100 40GB)

> Week 1 완료 후 Drive에 `exp1_hidden_states.npy`가 있어야 합니다.

## 0. 환경 세팅

In [ ]:
# GPU 확인
!nvidia-smi | head -5

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/self-correcting-llm'
HS_CACHE  = f'{DRIVE_DIR}/hs_cache'
os.environ['HS_CACHE'] = HS_CACHE

print(f'HS cache: {HS_CACHE}')

# 파일 존재 확인
from pathlib import Path
hs_path = Path(HS_CACHE) / 'exp1_hidden_states.npy'
if hs_path.exists():
    size_mb = hs_path.stat().st_size / 1e6
    print(f'exp1_hidden_states.npy 확인: {size_mb:.1f} MB')
else:
    print('exp1_hidden_states.npy 없음 — Week 1을 먼저 실행하세요')

In [ ]:
# 레포 클론 및 이동
import os
REPO = 'self-correcting-llm'
if os.path.exists('config.py'):
    print(f'레포 루트: {os.getcwd()}')
elif os.path.exists(REPO):
    %cd {REPO}
else:
    !git clone -b phase/2-probing-exp1 \
        https://github.com/uuyeong/self-correcting-llm.git
    %cd {REPO}

In [ ]:
!pip install -q -r requirements.txt
print('설치 완료')

In [ ]:
import sys
sys.path.insert(0, '.')
import config
print(f'Primary model : {config.PRIMARY_MODEL}')
print(f'BEST_LAYER    : {config.BEST_LAYER}  (None → Exp 1 실행 후 업데이트)')

---
## 1. Hidden States 로드 확인

In [ ]:
import numpy as np
from pathlib import Path

hs_full    = np.load(Path(HS_CACHE) / 'exp1_hidden_states.npy')
all_labels = np.load(Path(HS_CACHE) / 'exp1_labels.npy')

print(f'hidden_states : {hs_full.shape}  (N, n_layers, hidden_dim)')
print(f'labels        : {all_labels.shape}')
print(f'  truthful (0): {(all_labels==0).sum()}')
print(f'  hallucinated (1): {(all_labels==1).sum()}')

---
## 2. Exp 1 실행 — 레이어별 AUROC 분석

GPU 없이 CPU만으로 LogisticRegression 학습 → A100 불필요, 약 2~5분 소요

In [ ]:
%run experiments/exp1_layer_analysis.py

---
## 3. 결과 확인 및 히트맵 인라인 출력

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.image as mpimg
from pathlib import Path

# AUROC 수치 출력
with open('results/exp1_aurocs.json') as f:
    data = json.load(f)

aurocs = data['aurocs']
best_layer = data['best_layer']
best_auroc = data['best_auroc']

print(f'Best layer : {best_layer}')
print(f'Best AUROC : {best_auroc:.4f}')
print('\n전체 레이어 AUROC:')
for i, a in enumerate(aurocs):
    bar = '#' * int(a * 40)
    print(f'  Layer {i:2d}: {a:.4f} {bar}')

In [ ]:
# 히트맵 인라인 출력
from src.visualization.plots import plot_layer_auroc_heatmap
fig = plot_layer_auroc_heatmap(aurocs, model_name='LLaMA-3.1-8B', save=False)
plt.show()

---
## 4. config.py BEST_LAYER 업데이트

위 출력에서 `Best layer` 값을 확인하고 아래 셀의 `BEST_LAYER`에 입력하세요.

In [ ]:
import re, pathlib, json

# Exp 1 결과에서 자동으로 읽어옴
with open('results/exp1_aurocs.json') as f:
    BEST_LAYER = json.load(f)['best_layer']

cfg_path = pathlib.Path('config.py')
cfg_text = cfg_path.read_text()
cfg_text = re.sub(r'BEST_LAYER\s*=\s*\S+', f'BEST_LAYER = {BEST_LAYER}', cfg_text)
cfg_path.write_text(cfg_text)

print(f'config.py 업데이트 완료: BEST_LAYER = {BEST_LAYER}')

# 반영 확인
import importlib
importlib.reload(config)
print(f'config.BEST_LAYER = {config.BEST_LAYER}')

---
## 5. Week 2 완료 체크리스트

In [ ]:
from pathlib import Path
import json, importlib
importlib.reload(config)

with open('results/exp1_aurocs.json') as f:
    auroc_data = json.load(f)

checks = [
    ('exp1_aurocs.json 저장',             Path('results/exp1_aurocs.json').exists()),
    ('exp1_layer_auroc_heatmap.pdf 생성', Path('results/figures/exp1_layer_auroc_heatmap.pdf').exists()),
    ('exp1_probing_result.pkl (Drive)',    (Path(HS_CACHE) / 'exp1_probing_result.pkl').exists()),
    ('Best AUROC > 0.60',                 auroc_data['best_auroc'] > 0.60),
    ('config.BEST_LAYER 업데이트',        config.BEST_LAYER is not None),
]

all_pass = True
for name, ok in checks:
    mark = 'v' if ok else 'x'
    print(f'  [{mark}] {name}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('Week 2 완료! phase/3-regen-pipeline-exp2-3 브랜치로 이동 준비')
else:
    print('미완료 항목 있음')

---
## 6. Git 커밋 & 푸시

```bash
git add experiments/exp1_layer_analysis.py \
        results/exp1_aurocs.json \
        results/figures/exp1_layer_auroc_heatmap.pdf \
        config.py \
        notebooks/week2_probing_exp1.ipynb
git commit -m "feat(exp1): 레이어별 AUROC 분석 완료, BEST_LAYER 업데이트"
git push origin phase/2-probing-exp1
```